In [1]:
import rasterio
import numpy as np
import geopandas as gpd
import pandas as pd
from rasterio.mask import mask
from shapely.geometry import mapping
import os

In [2]:
# ════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════
CROP_MAP_FILE = r'D:\University of Phayao\เอกสารการเรียน\Paper 2\Revise\ML\crop_map_v3b_2020.tif' 
ZONE_A_FILE   = r'D:\University of Phayao\เอกสารการเรียน\Paper 2\Revise\ML\zone_a_rainfed.shp'
ZONE_B_FILE   = r'D:\University of Phayao\เอกสารการเรียน\Paper 2\Revise\ML\zone_b_irrigated.shp'
OUT_DIR       = r'D:\University of Phayao\เอกสารการเรียน\Paper 2\Revise\ML\output'

CLASS_LABELS  = {0:'rice', 1:'corn', 2:'cassava', 3:'longan', 4:'etc'}
NODATA_VAL    = 255   # จาก inspect เดิม

In [3]:
# ════════════════════════════════════════════════════════
# STEP 1 — โหลด zones
# ════════════════════════════════════════════════════════
zone_a = gpd.read_file(ZONE_A_FILE)
zone_b = gpd.read_file(ZONE_B_FILE)

with rasterio.open(CROP_MAP_FILE) as src:
    crop_crs = src.crs
    pixel_area_m2 = abs(src.res[0] * src.res[1])  # 20×20 = 400 m²

print(f"Pixel area: {pixel_area_m2} m² ({pixel_area_m2/10000:.4f} ha)")

# Reproject zones ให้ตรงกับ crop map (ควรเป็น EPSG:32647 เหมือนกัน)
zone_a = zone_a.to_crs(crop_crs)
zone_b = zone_b.to_crs(crop_crs)

Pixel area: 400.0 m² (0.0400 ha)


In [4]:
# ════════════════════════════════════════════════════════
# STEP 2 — Extract crop pixels ต่อ zone
# ════════════════════════════════════════════════════════
def extract_crop_area(zone_gdf, zone_name, crop_tif, class_labels, nodata=255):
    """Mask raster ด้วย zone polygon → นับ pixels ต่อ class"""
    geoms = [mapping(g) for g in zone_gdf.geometry if g is not None]

    with rasterio.open(crop_tif) as src:
        masked, _ = mask(src, geoms, crop=True, filled=True,
                         nodata=nodata, all_touched=False)
        data = masked[0]

    total_valid = np.sum(data != nodata)
    rows = []
    for val, name in class_labels.items():
        n_px   = int(np.sum(data == val))
        area_m2 = n_px * pixel_area_m2
        rows.append({
            'zone'      : zone_name,
            'crop_id'   : val,
            'crop'      : name,
            'n_pixels'  : n_px,
            'area_m2'   : area_m2,
            'area_ha'   : round(area_m2 / 10000, 2),
            'area_km2'  : round(area_m2 / 1e6,   4),
            'pct_zone'  : round(n_px / total_valid * 100, 1) if total_valid > 0 else 0,
        })
    return pd.DataFrame(rows), total_valid

print("\n=== STEP 2 — Extract crop area per zone ===")
df_a, valid_a = extract_crop_area(zone_a, 'A', CROP_MAP_FILE, CLASS_LABELS)
df_b, valid_b = extract_crop_area(zone_b, 'B', CROP_MAP_FILE, CLASS_LABELS)

crop_area = pd.concat([df_a, df_b], ignore_index=True)


=== STEP 2 — Extract crop area per zone ===


In [5]:
# ════════════════════════════════════════════════════════
# STEP 3 — Sanity Check
# ════════════════════════════════════════════════════════
print("\n=== STEP 3 — Sanity Check ===")
print("\nCrop area per zone (ha):")
pivot = crop_area.pivot_table(
    index='crop', columns='zone',
    values=['area_ha','pct_zone'], aggfunc='sum'
).round(2)
print(pivot.to_string())

# ตรวจ total area สมเหตุสมผลไหม
total_a_ha = df_a['area_ha'].sum()
total_b_ha = df_b['area_ha'].sum()
print(f"\nZone A valid crop pixels: {valid_a:,} → {total_a_ha:.1f} ha")
print(f"Zone B valid crop pixels: {valid_b:,} → {total_b_ha:.1f} ha")
print(f"Zone A from shapefile   : {zone_a.geometry.area.sum()/10000:.1f} ha")
print(f"Zone B from shapefile   : {zone_b.geometry.area.sum()/10000:.1f} ha")

# ตรวจ dominant crop ต่อ zone (ควรสมเหตุสมผลตามภูมิศาสตร์)
print("\nDominant crop per zone:")
for zn, grp in crop_area.groupby('zone'):
    top = grp.nlargest(1, 'area_ha').iloc[0]
    print(f"  Zone {zn}: {top['crop']} ({top['area_ha']:.1f} ha, {top['pct_zone']:.1f}%)")


=== STEP 3 — Sanity Check ===

Crop area per zone (ha):
         area_ha         pct_zone      
zone           A       B        A     B
crop                                   
cassava     0.16    0.32      0.0   0.0
corn      621.36  215.52     22.6  29.6
etc       156.68   57.88      5.7   8.0
longan    461.36  170.64     16.8  23.5
rice     1510.72  282.88     54.9  38.9

Zone A valid crop pixels: 68,757 → 2750.3 ha
Zone B valid crop pixels: 18,181 → 727.2 ha
Zone A from shapefile   : 3754.4 ha
Zone B from shapefile   : 876.0 ha

Dominant crop per zone:
  Zone A: rice (1510.7 ha, 54.9%)
  Zone B: rice (282.9 ha, 38.9%)


In [6]:
# ════════════════════════════════════════════════════════
# STEP 4 — สร้าง area_dict สำหรับใช้ใน demand calculation
# ════════════════════════════════════════════════════════
# Format ที่ Step 13 ต้องการ: {'rice': m², 'corn': m², ...}
area_zone_a = dict(zip(df_a['crop'], df_a['area_m2']))
area_zone_b = dict(zip(df_b['crop'], df_b['area_m2']))

print("\narea_zone_a (m²):")
for k, v in area_zone_a.items():
    print(f"  '{k}': {v:,.0f}")

print("\narea_zone_b (m²):")
for k, v in area_zone_b.items():
    print(f"  '{k}': {v:,.0f}")


area_zone_a (m²):
  'rice': 15,107,200
  'corn': 6,213,600
  'cassava': 1,600
  'longan': 4,613,600
  'etc': 1,566,800

area_zone_b (m²):
  'rice': 2,828,800
  'corn': 2,155,200
  'cassava': 3,200
  'longan': 1,706,400
  'etc': 578,800


In [7]:
# ════════════════════════════════════════════════════════
# STEP 5 — Export
# ════════════════════════════════════════════════════════
crop_area.to_csv(os.path.join(OUT_DIR, 'crop_area_per_zone.csv'), index=False)
print(f"\n✅ Saved: crop_area_per_zone.csv")
print(f"   Rows: {len(crop_area)} | Columns: {crop_area.columns.tolist()}")
print("\n📌 NEXT: Step 13 → คำนวณ NIR_A และ GIR_B รายสัปดาห์")


✅ Saved: crop_area_per_zone.csv
   Rows: 10 | Columns: ['zone', 'crop_id', 'crop', 'n_pixels', 'area_m2', 'area_ha', 'area_km2', 'pct_zone']

📌 NEXT: Step 13 → คำนวณ NIR_A และ GIR_B รายสัปดาห์


In [10]:
import pandas as pd
import numpy as np
import os

# ════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════
CLIMATE_FILE  = r'climate_weekly_phayao_2020_2024.csv'
KC_FILE       = r'kc_weekly_lookup_all_crops.csv'
OUT_DIR       = r'D:\University of Phayao\เอกสารการเรียน\Paper 2\Revise\ML\output'

# crop area จาก Step 12 (m²) — แทนด้วยค่าจริง
AREA_ZONE_A = {
    'rice'   : 1510.72 * 10000,
    'corn'   : 621.36  * 10000,
    'longan' : 461.36  * 10000,
    'cassava': 0.16    * 10000,
    'etc'    : 156.68  * 10000,
}
AREA_ZONE_B = {
    'rice'   : 282.88  * 10000,
    'corn'   : 215.52  * 10000,
    'longan' : 170.64  * 10000,
    'cassava': 0.32    * 10000,
    'etc'    : 57.88   * 10000,
}

IE = 0.75   # Irrigation efficiency (FAO-56 surface irrigation)

In [11]:
# ════════════════════════════════════════════════════════
# STEP 1 — โหลดข้อมูล
# ════════════════════════════════════════════════════════
print("="*55)
print("STEP 1 — โหลดข้อมูล")
print("="*55)

climate = pd.read_csv(CLIMATE_FILE)
kc      = pd.read_csv(KC_FILE)

print(f"climate: {len(climate)} weeks | cols: {climate.columns.tolist()}")
print(f"kc     : {len(kc)} rows     | cols: {kc.columns.tolist()}")
print(f"kc crops: {kc['crop'].unique()}")

STEP 1 — โหลดข้อมูล
climate: 260 weeks | cols: ['year', 'week', 'ET0_mm_week', 'T_mean', 'RH_pct', 'VPD_kPa', 'u2_ms', 'Rn_MJ', 'P_mm_week', 'P_eff_paddy', 'P_eff_upland', 'deficit_paddy_mm', 'deficit_upland_mm', 'P_4week', 'SPI_4', 'drought_flag', 'AI_week', 'imputed', 'n_days']
kc     : 260 rows     | cols: ['crop', 'week', 'stage', 'Kc']
kc crops: <StringArray>
['rice', 'corn', 'cassava', 'longan', 'rubber']
Length: 5, dtype: str


In [12]:
# ════════════════════════════════════════════════════════
# STEP 2 — สร้าง Kc lookup (year, week, crop)
# ════════════════════════════════════════════════════════
# Kc อาจมีแค่ crop+week (ไม่มี year) → ใช้ mean ข้ามปี
# หรือมี year ด้วย → merge ตรง
kc_cols = kc.columns.tolist()
print(f"\nKc columns: {kc_cols}")

if 'year' in kc_cols:
    kc_lookup = kc[['year','week','crop','Kc']].copy()
    merge_keys = ['year','week','crop']
else:
    # ใช้ค่า Kc เฉลี่ยต่อ week ข้ามปี
    kc_lookup = kc.groupby(['week','crop'])['Kc'].mean().reset_index()
    merge_keys = ['week','crop']

print(f"Kc lookup shape: {kc_lookup.shape}")
print(f"Sample Kc values:")
print(kc_lookup.head(8).to_string(index=False))



Kc columns: ['crop', 'week', 'stage', 'Kc']
Kc lookup shape: (260, 3)
Sample Kc values:
 week    crop    Kc
    1 cassava 1.050
    1    corn 0.000
    1  longan 0.997
    1    rice 0.000
    1  rubber 0.630
    2 cassava 1.050
    2    corn 0.000
    2  longan 0.997


In [13]:
# ════════════════════════════════════════════════════════
# STEP 3 — คำนวณ NIR_A และ GIR_B รายสัปดาห์
# ════════════════════════════════════════════════════════
print("\n" + "="*55)
print("STEP 3 — คำนวณ NIR_A / GIR_B")
print("="*55)

results = []

for _, row in climate.iterrows():
    yr      = int(row['year'])
    wk      = int(row['week'])
    et0     = float(row['ET0_mm_week'])
    p_eff_a = float(row['P_eff_upland'])  # Zone A = rainfed → upland P_eff
    p_eff_b = float(row['P_eff_paddy'])   # Zone B = irrigated (rice-dominant)

    # --- Zone A: NIR (Net Irrigation Requirement) ---
    nir_a_total = 0.0
    etc_a_total = 0.0
    for crop, area_m2 in AREA_ZONE_A.items():
        if area_m2 == 0:
            continue
        # หา Kc สำหรับ crop นี้ที่ week นี้
        if 'year' in merge_keys:
            kc_val = kc_lookup[(kc_lookup['year']==yr) &
                               (kc_lookup['week']==wk) &
                               (kc_lookup['crop']==crop)]['Kc']
        else:
            kc_val = kc_lookup[(kc_lookup['week']==wk) &
                               (kc_lookup['crop']==crop)]['Kc']

        kc_val = float(kc_val.values[0]) if len(kc_val) > 0 else 0.0
        etc_mm = kc_val * et0                          # mm/week
        nir_mm = max(0.0, etc_mm - p_eff_a)            # mm/week
        nir_a_total += nir_mm * area_m2 / 1000         # m³/week
        etc_a_total += etc_mm * area_m2 / 1000

    # --- Zone B: GIR (Gross Irrigation Requirement) ---
    gir_b_total = 0.0
    etc_b_total = 0.0
    for crop, area_m2 in AREA_ZONE_B.items():
        if area_m2 == 0:
            continue
        if 'year' in merge_keys:
            kc_val = kc_lookup[(kc_lookup['year']==yr) &
                               (kc_lookup['week']==wk) &
                               (kc_lookup['crop']==crop)]['Kc']
        else:
            kc_val = kc_lookup[(kc_lookup['week']==wk) &
                               (kc_lookup['crop']==crop)]['Kc']

        kc_val = float(kc_val.values[0]) if len(kc_val) > 0 else 0.0
        etc_mm  = kc_val * et0
        nir_mm  = max(0.0, etc_mm - p_eff_b)
        gir_mm  = nir_mm / IE                          # หาร IE
        gir_b_total += gir_mm * area_m2 / 1000
        etc_b_total += etc_mm * area_m2 / 1000

    results.append({
        'year'         : yr,
        'week'         : wk,
        'ET0_mm_week'  : round(et0, 3),
        'P_eff_upland' : round(p_eff_a, 3),
        'P_eff_paddy'  : round(p_eff_b, 3),
        'ETc_A_m3'     : round(etc_a_total, 1),
        'NIR_A_m3'     : round(nir_a_total, 1),
        'ETc_B_m3'     : round(etc_b_total, 1),
        'GIR_B_m3'     : round(gir_b_total, 1),
        'D_total_m3'   : round(nir_a_total + gir_b_total, 1),
        'SPI_4'        : float(row['SPI_4']),
        'drought_flag' : int(row['drought_flag']),
    })

demand = pd.DataFrame(results)


STEP 3 — คำนวณ NIR_A / GIR_B


In [14]:
# ════════════════════════════════════════════════════════
# STEP 4 — Sanity Check
# ════════════════════════════════════════════════════════
print("\n" + "="*55)
print("STEP 4 — Sanity Check")
print("="*55)

print(f"Total weeks: {len(demand)}")
print(f"\nStats (m³/week):")
print(demand[['NIR_A_m3','GIR_B_m3','D_total_m3']].describe().round(0).to_string())

# ตรวจ negative
neg = (demand['D_total_m3'] < 0).sum()
print(f"\nNegative D_total: {neg}  (ควร = 0)")

# Seasonal pattern
demand['season'] = demand['week'].apply(
    lambda w: 'Dry-hot(10-18)' if 10<=w<=18
    else ('Wet(19-44)' if 19<=w<=44 else 'Dry-cool'))
print("\nค่าเฉลี่ยตามฤดูกาล (m³/week):")
print(demand.groupby('season')[['NIR_A_m3','GIR_B_m3','D_total_m3']]
      .mean().round(0).to_string())

# Annual demand
print("\nAnnual demand (m³/year):")
print(demand.groupby('year')['D_total_m3'].sum()
      .apply(lambda x: f"{x/1e6:.3f} Mm³").to_string())

# Sample
print("\nตัวอย่าง wet season (W26-30, 2022):")
s = demand[(demand['year']==2022) & (demand['week'].between(26,30))]
print(s[['year','week','ET0_mm_week','NIR_A_m3',
         'GIR_B_m3','D_total_m3','SPI_4']].to_string(index=False))

print("\nตัวอย่าง dry season (W10-14, 2023):")
s2 = demand[(demand['year']==2023) & (demand['week'].between(10,14))]
print(s2[['year','week','ET0_mm_week','NIR_A_m3',
          'GIR_B_m3','D_total_m3','SPI_4']].to_string(index=False))



STEP 4 — Sanity Check
Total weeks: 260

Stats (m³/week):
       NIR_A_m3  GIR_B_m3  D_total_m3
count     260.0     260.0       260.0
mean    78650.0   27329.0    105978.0
std    105020.0   32229.0    135294.0
min         0.0       0.0         0.0
25%         0.0       0.0         0.0
50%     44405.0   12729.0     58852.0
75%    113452.0   51383.0    165611.0
max    523806.0  164834.0    655571.0

Negative D_total: 0  (ควร = 0)

ค่าเฉลี่ยตามฤดูกาล (m³/week):
                NIR_A_m3  GIR_B_m3  D_total_m3
season                                        
Dry-cool         78358.0   32166.0    110524.0
Dry-hot(10-18)   84968.0   39142.0    124110.0
Wet(19-44)       76654.0   20076.0     96730.0

Annual demand (m³/year):
year
2020    5.453 Mm³
2021    4.634 Mm³
2022    4.380 Mm³
2023    7.055 Mm³
2024    6.032 Mm³

ตัวอย่าง wet season (W26-30, 2022):
 year  week  ET0_mm_week  NIR_A_m3  GIR_B_m3  D_total_m3  SPI_4
 2022    26        20.23       0.0       0.0         0.0 -0.018
 2022    27     

In [15]:
# ════════════════════════════════════════════════════════
# STEP 5 — Export
# ════════════════════════════════════════════════════════
out_path = os.path.join(OUT_DIR, 'water_demand_weekly_dual_zone.csv')
demand.to_csv(out_path, index=False)
print(f"\n✅ Saved: water_demand_weekly_dual_zone.csv")
print(f"   Rows: {len(demand)} | Columns: {demand.columns.tolist()}")
print("\n📌 NEXT: Phase 4 — Feature engineering → ML model")


✅ Saved: water_demand_weekly_dual_zone.csv
   Rows: 260 | Columns: ['year', 'week', 'ET0_mm_week', 'P_eff_upland', 'P_eff_paddy', 'ETc_A_m3', 'NIR_A_m3', 'ETc_B_m3', 'GIR_B_m3', 'D_total_m3', 'SPI_4', 'drought_flag', 'season']

📌 NEXT: Phase 4 — Feature engineering → ML model
